# CG-DETR Inference on QVHighlights

Runs evaluation with the pretrained CG-DETR checkpoint on the QVHighlights val set.

**Before running:** download the pretrained checkpoint from the [Model Zoo](https://drive.google.com/drive/folders/1_hEqXbvDv4AyEn5unyn_kE784ruqrzEJ) and place it at the path set in `CKPT_PATH` below.

## 0. Setup

In [1]:
import sys
import os

# Add CGDETR to path so its modules are importable
CGDETR_ROOT = "/projectnb/cs585/projects/VMR/vmr_project/CGDETR"
if CGDETR_ROOT not in sys.path:
    sys.path.insert(0, CGDETR_ROOT)

# Run all relative imports from within CGDETR_ROOT
os.chdir(CGDETR_ROOT)
print("Working directory:", os.getcwd())

Working directory: /projectnb/cs585/projects/VMR/vmr_project/CGDETR


## 1. Paths — edit CKPT_PATH before running

In [2]:
PROJECT_ROOT = "/projectnb/cs585/projects/VMR/vmr_project"

# --- EDIT THIS: path to the downloaded pretrained checkpoint (.ckpt) ---
CKPT_PATH = f"{PROJECT_ROOT}/checkpoints/cgdetr/model_best.ckpt"

# Feature directories (verified to have correct keys and shapes)
V_FEAT_DIRS = [
    f"{PROJECT_ROOT}/data/qvhighlights/features/slowfast",
    f"{PROJECT_ROOT}/data/qvhighlights/features/clip",
]
T_FEAT_DIR  = f"{PROJECT_ROOT}/data/qvhighlights/txt_features/clip_text"

# Annotation files (already present in CGDETR/data/)
EVAL_SPLIT   = "val"   # change to "test" for test-set submission
EVAL_PATH    = f"{CGDETR_ROOT}/data/highlight_{EVAL_SPLIT}_release.jsonl"

# Where to write results
RESULTS_ROOT = f"{PROJECT_ROOT}/results/cgdetr"
os.makedirs(RESULTS_ROOT, exist_ok=True)

print("Checkpoint :", CKPT_PATH)
print("Val annotations:", EVAL_PATH)
print("Video feat dirs:", V_FEAT_DIRS)
print("Text feat dir  :", T_FEAT_DIR)

Checkpoint : /projectnb/cs585/projects/VMR/vmr_project/checkpoints/cgdetr/model_best.ckpt
Val annotations: /projectnb/cs585/projects/VMR/vmr_project/CGDETR/data/highlight_val_release.jsonl
Video feat dirs: ['/projectnb/cs585/projects/VMR/vmr_project/data/qvhighlights/features/slowfast', '/projectnb/cs585/projects/VMR/vmr_project/data/qvhighlights/features/clip']
Text feat dir  : /projectnb/cs585/projects/VMR/vmr_project/data/qvhighlights/txt_features/clip_text


## 2. Build options

`TestOptions` automatically loads the saved `opt.json` sitting next to the checkpoint and uses it to restore all model hyper-parameters. We then override the feature-path fields so they point to our SCC data layout instead of the original author paths.

In [3]:
import sys

# Simulate the CLI arguments that inference.sh would pass
sys.argv = [
    "inference.py",
    "--resume",       CKPT_PATH,
    "--eval_split_name", EVAL_SPLIT,
    "--eval_path",    EVAL_PATH,
    "--results_root", RESULTS_ROOT,
    "--v_feat_dirs",  *V_FEAT_DIRS,
    "--t_feat_dir",   T_FEAT_DIR,
]

from cg_detr.config import TestOptions

opt = TestOptions().parse()

# TestOptions restores v_feat_dirs / t_feat_dir from the saved opt.json;
# override them here to point to our SCC paths.
opt.v_feat_dirs = V_FEAT_DIRS
opt.t_feat_dir  = T_FEAT_DIR
opt.eval_path   = EVAL_PATH
opt.results_dir = RESULTS_ROOT
# The Lighthouse checkpoint was trained with TEF already included in v_feat_dim=2818.
# CGDETR's config adds +2 for TEF on top, causing a mismatch — override it back.
opt.v_feat_dim  = 2818

print("\nEffective feature dirs:")
for d in opt.v_feat_dirs:
    print(" ", d)
print("Text feat dir:", opt.t_feat_dir)
print("v_feat_dim   :", opt.v_feat_dim)
print("Device:", opt.device)

|                             | 0                                                                                                                        |
|:----------------------------|:-------------------------------------------------------------------------------------------------------------------------|
| dset_name                   | hl                                                                                                                       |
| dset_domain                 |                                                                                                                          |
| eval_split_name             | val                                                                                                                      |
| debug                       | False                                                                                                                    |
| data_ratio                  | 1.0                                   

## 3. Build dataset

In [4]:
from cg_detr.start_end_dataset import StartEndDataset

eval_dataset = StartEndDataset(
    dset_name=opt.dset_name,
    data_path=opt.eval_path,
    v_feat_dirs=opt.v_feat_dirs,
    q_feat_dir=opt.t_feat_dir,
    q_feat_type="last_hidden_state",
    max_q_l=opt.max_q_l,
    max_v_l=opt.max_v_l,
    ctx_mode=opt.ctx_mode,
    data_ratio=opt.data_ratio,
    normalize_v=not opt.no_norm_vfeat,
    normalize_t=not opt.no_norm_tfeat,
    clip_len=opt.clip_length,
    max_windows=opt.max_windows,
    load_labels=True,   # val set has GT labels
    span_loss_type=opt.span_loss_type,
    txt_drop_ratio=0,
    dset_domain=opt.dset_domain,
)

print(f"Val set size: {len(eval_dataset)} examples")

Val set size: 1550 examples


## 4. Load model from checkpoint

In [5]:
from cg_detr.inference import setup_model

model, criterion, _, _ = setup_model(opt)
model.eval()
print("Model loaded successfully.")

2026-04-05 18:24:59.084:INFO:cg_detr.inference - setup model/optimizer/scheduler
2026-04-05 18:24:59.480:INFO:cg_detr.inference - CUDA enabled.
/projectnb/cs585/projects/VMR/envs/vmr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-05 18:25:00.024:INFO:cg_detr.inference - Load checkpoint from /projectnb/cs585/projects/VMR/vmr_project/checkpoints/cgdetr/model_best.ckpt
2026-04-05 18:25:00.722:INFO:cg_detr.inference - Loaded model saved at epoch 129 from checkpoint: /projectnb/cs585/projects/VMR/vmr_project/checkpoints/cgdetr/model_best.ckpt


Model loaded successfully.


## 5. Run inference

In [6]:
import torch
import torch.backends.cudnn as cudnn
from cg_detr.inference import eval_epoch

cudnn.benchmark = True
cudnn.deterministic = False

save_filename = f"hl_{EVAL_SPLIT}_submission.jsonl"

with torch.no_grad():
    metrics, metrics_nms, loss_meters, output_files = eval_epoch(
        model, eval_dataset, opt,
        save_submission_filename=save_filename,
        criterion=criterion,
    )

print("\nOutput files:", output_files)

2026-04-05 18:25:02.268:INFO:cg_detr.inference - Generate submissions
convert to multiples of clip_length=2: 100%|██████████| 1550/1550 [00:00<00:00, 39547.48it/s]
2026-04-05 18:25:08.435:INFO:cg_detr.inference - Saving/Evaluating before nms results


short: [0, 10], 429/1550=27.68 examples.
middle: [10, 30], 957/1550=61.74 examples.
long: [30, 150], 574/1550=37.03 examples.
full: [0, 150], 1550/1550=100.00 examples.

Output files: ['/projectnb/cs585/projects/VMR/vmr_project/results/cgdetr/hl_val_submission.jsonl', '/projectnb/cs585/projects/VMR/vmr_project/results/cgdetr/hl_val_submission_metrics.json']


## 6. Results

In [7]:
# Metrics required by the proposal (Section 3.3)
# - Moment Retrieval : Recall@1 at IoU = 0.3, 0.5, 0.7
# - Highlight Detection: mAP, HIT@1

if metrics is None:
    print("No GT available for test split — switch EVAL_SPLIT to 'val' to see metrics.")
else:
    brief = metrics["brief"]

    # Moment Retrieval keys (from standalone_eval/eval.py)
    mr_r1_03 = brief.get("MR-full-R1@0.3", None)
    mr_r1_05 = brief.get("MR-full-R1@0.5", None)
    mr_r1_07 = brief.get("MR-full-R1@0.7", None)
    mr_map   = brief.get("MR-full-mAP",    None)

    # Highlight Detection keys (pattern: "HL-<score_name>-mAP" / "HL-<score_name>-Hit1")
    hl_map  = next((v for k, v in brief.items() if k.startswith("HL-") and k.endswith("-mAP")),  None)
    hl_hit1 = next((v for k, v in brief.items() if k.startswith("HL-") and k.endswith("-Hit1")), None)

    print("=" * 52)
    print(f"{'Metric':<30} {'Ours':>8}  {'Paper':>8}")
    print("-" * 52)
    print(f"{'MR  Recall@1 IoU=0.3':<30} {f'{mr_r1_03:.2f}' if mr_r1_03 is not None else 'N/A':>8}  {'--':>8}")
    print(f"{'MR  Recall@1 IoU=0.5':<30} {f'{mr_r1_05:.2f}' if mr_r1_05 is not None else 'N/A':>8}  {'56.56':>8}")
    print(f"{'MR  Recall@1 IoU=0.7':<30} {f'{mr_r1_07:.2f}' if mr_r1_07 is not None else 'N/A':>8}  {'38.97':>8}")
    print(f"{'MR  mAP (avg)':<30} {f'{mr_map:.2f}'   if mr_map   is not None else 'N/A':>8}  {'38.22':>8}")
    print("-" * 52)
    print(f"{'HD  mAP':<30} {f'{hl_map:.2f}'  if hl_map  is not None else 'N/A':>8}  {'39.19':>8}")
    print(f"{'HD  HIT@1':<30} {f'{hl_hit1:.2f}' if hl_hit1 is not None else 'N/A':>8}  {'60.48':>8}")
    print("=" * 52)

    print("\n--- Full brief dict (all keys) ---")
    import pprint
    pprint.pprint(brief)

Metric                             Ours     Paper
----------------------------------------------------
MR  Recall@1 IoU=0.3              77.94        --
MR  Recall@1 IoU=0.5              67.35     56.56
MR  Recall@1 IoU=0.7              52.06     38.97
MR  mAP (avg)                     44.93     38.22
----------------------------------------------------
HD  mAP                           77.78     39.19
HD  HIT@1                         80.13     60.48

--- Full brief dict (all keys) ---
OrderedDict([('MR-full-R1@0.3', 77.94),
             ('MR-full-R1@0.5', 67.35),
             ('MR-full-R1@0.7', 52.06),
             ('MR-full-mAP', 44.93),
             ('MR-full-mAP@0.5', 65.57),
             ('MR-full-mAP@0.75', 45.73),
             ('MR-full-mIoU', 61.69),
             ('MR-long-mAP', 49.36),
             ('MR-middle-mAP', 48.3),
             ('MR-short-mAP', 10.58),
             ('HL-min-Fair-mAP', 77.78),
             ('HL-min-Fair-Hit1', 80.13),
             ('HL-min-Good-mAP', 6

In [8]:
# After-NMS results (only printed if the checkpoint had nms_thd set)
if metrics_nms:
    brief_nms = metrics_nms["brief"]

    mr_r1_03 = brief_nms.get("MR-full-R1@0.3", None)
    mr_r1_05 = brief_nms.get("MR-full-R1@0.5", None)
    mr_r1_07 = brief_nms.get("MR-full-R1@0.7", None)
    mr_map   = brief_nms.get("MR-full-mAP",    None)
    hl_map   = next((v for k, v in brief_nms.items() if k.startswith("HL-") and k.endswith("-mAP")),  None)
    hl_hit1  = next((v for k, v in brief_nms.items() if k.startswith("HL-") and k.endswith("-Hit1")), None)

    print("=" * 52)
    print("After NMS")
    print(f"{'Metric':<30} {'Ours':>8}  {'Paper':>8}")
    print("-" * 52)
    print(f"{'MR  Recall@1 IoU=0.3':<30} {f'{mr_r1_03:.2f}' if mr_r1_03 is not None else 'N/A':>8}  {'--':>8}")
    print(f"{'MR  Recall@1 IoU=0.5':<30} {f'{mr_r1_05:.2f}' if mr_r1_05 is not None else 'N/A':>8}  {'56.56':>8}")
    print(f"{'MR  Recall@1 IoU=0.7':<30} {f'{mr_r1_07:.2f}' if mr_r1_07 is not None else 'N/A':>8}  {'38.97':>8}")
    print(f"{'MR  mAP (avg)':<30} {f'{mr_map:.2f}'   if mr_map   is not None else 'N/A':>8}  {'38.22':>8}")
    print("-" * 52)
    print(f"{'HD  mAP':<30} {f'{hl_map:.2f}'  if hl_map  is not None else 'N/A':>8}  {'39.19':>8}")
    print(f"{'HD  HIT@1':<30} {f'{hl_hit1:.2f}' if hl_hit1 is not None else 'N/A':>8}  {'60.48':>8}")
    print("=" * 52)
else:
    print("NMS not applied (nms_thd=-1 in checkpoint options).")

NMS not applied (nms_thd=-1 in checkpoint options).
